# 🔢 Python Math & Number Theory — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Number theory problems share a common theme: find hidden structure by breaking numbers apart. The Sieve is a stencil — stamp out all multiples, whatever's left is prime. GCD is a shrinking-ruler — keep halving the measurement gap until it fits perfectly. Bezout's theorem says two jugs can reach any volume that the ruler fits into. Base conversion is repeated remaindering — peel off the last digit, shift right, repeat. In every case: look for the divisibility pattern, and the hard problem melts away.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Math & Number Theory? The Visual Model](#1) |
| 2 | [Creating / Setup — Core Math Primitives](#2) |
| 3 | [The Core API — GCD, LCM, Modular, Power](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Sieve of Eratosthenes (LC 204)](#5) |
| 6 | [Pattern 2: Max Points on a Line (LC 149)](#6) |
| 7 | [Pattern 3: Water and Jug Problem (LC 365)](#7) |
| 8 | [Pattern 4: Excel Sheet Column Title (LC 168)](#8) |
| 9 | [Pattern 5: Fast Power & Modular Arithmetic](#9) |
| 10 | [The Math & Number Theory Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. 🗺️ What Is Math & Number Theory? The Visual Model

```
SIEVE OF ERATOSTHENES — find all primes up to 20

  Start:  [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
  p=2:    cross [4,6,8,10,12,14,16,18,20]
  p=3:    cross [9,15]  (6 already crossed)
  p=5:    cross [25>20] → stop when p²>n
  Primes: [2, 3, 5, 7, 11, 13, 17, 19]

  KEY: start crossing at p² (everything before was crossed by smaller primes)
  TIME: O(n log log n) — harmonic series of prime reciprocals

────────────────────────────────────────────────────────────────
EUCLIDEAN GCD — gcd(48, 18)

  gcd(48, 18)  →  gcd(18, 48 % 18 = 12)
  gcd(18, 12)  →  gcd(12, 18 % 12 = 6)
  gcd(12,  6)  →  gcd(6,  12 % 6  = 0)
  gcd(6,   0)  →  6  (b==0 → return a)

  Physical analogy: measuring 48cm with 18cm ruler, keep taking remainders
  until the ruler fits perfectly — that size is the GCD.

────────────────────────────────────────────────────────────────
BEZOUT'S THEOREM

  ax + by = c  has integer solution  ↔  gcd(a, b) divides c

  Two jugs, sizes 3 and 5:
    gcd(3,5)=1  →  can reach ANY integer volume 0..8
  Two jugs, sizes 4 and 6:
    gcd(4,6)=2  →  can only reach even volumes: 0,2,4,6,8,10
```

<a id='2'></a>
## 2. 🔧 Creating / Setup — Core Math Primitives

In [ ]:
import math

# GCD — greatest common divisor (Python 3.5+ has math.gcd)
print("GCD examples:")
print(f"  gcd(48, 18)  = {math.gcd(48, 18)}")     # 6
print(f"  gcd(100, 75) = {math.gcd(100, 75)}")    # 25
print(f"  gcd(7, 5)    = {math.gcd(7, 5)}")        # 1 (coprime)
print(f"  gcd(0, 5)    = {math.gcd(0, 5)}")        # 5 (gcd with 0 = the other)

# Manual GCD (Euclidean algorithm) for interviews without math import
def gcd(a, b):
    while b:
        a, b = b, a % b    # remainder shrinks until b=0; then a is the GCD
    return a

print(f"\nManual gcd(48,18) = {gcd(48, 18)}")      # 6

# LCM — least common multiple
def lcm(a, b):
    return a * b // math.gcd(a, b)   # avoid overflow: a//gcd * b also fine

print(f"lcm(4, 6)  = {lcm(4, 6)}")                 # 12
print(f"lcm(7, 5)  = {lcm(7, 5)}")                 # 35 (coprime → lcm = product)

# Modular arithmetic — stay in bounds for large exponents
MOD = 10**9 + 7
print(f"\n2^100 mod {MOD} = {pow(2, 100, MOD)}")   # Python pow(base,exp,mod) is O(log exp)

# Primality test — O(sqrt(n))
def is_prime(n):
    if n < 2: return False
    if n == 2: return True
    if n % 2 == 0: return False
    for i in range(3, int(n**0.5) + 1, 2):  # only odd factors up to sqrt(n)
        if n % i == 0:
            return False
    return True

print(f"\nis_prime tests:")
for n in [1, 2, 3, 4, 17, 18, 97, 100]:
    print(f"  is_prime({n:3d}) = {is_prime(n)}")

print("Core math primitives defined.")

<a id='3'></a>
## 3. ⚡ The Core API — GCD, LCM, Modular, Power

```
OPERATION                     COMPLEXITY   WHAT IT DOES
──────────────────────────────────────────────────────────────────────
math.gcd(a, b)                O(log min)   greatest common divisor
a * b // math.gcd(a, b)       O(log min)   least common multiple
pow(base, exp, mod)           O(log exp)   modular fast exponentiation
n % k                         O(1)         remainder — core of all divisibility
n // k                        O(1)         integer division — digit extraction
n & (-n)                      O(1)         lowest set bit (lowbit — used in BIT)
int(n**0.5)                   O(1)         floor sqrt — trial division bound
sorted(set(nums))             O(n log n)   coordinate compression base
──────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  n ** exp for large exp without mod — result has exponential digit count
✅  pow(n, exp, MOD) — Python's built-in fast modular exponentiation
❌  check all i up to n for primality — trial division stops at sqrt(n)
✅  for i in range(2, int(n**0.5)+1): if n%i==0: return False
❌  sieve starting crossings at 2*p — start at p² (2p already crossed by 2)
✅  for j in range(p*p, n, p): sieve[j] = False
❌  slope as float (y2-y1)/(x2-x1) — floating point errors cause wrong grouping
✅  slope as reduced fraction (dy//g, dx//g) using GCD
```

In [ ]:
# LIVE DEMO: why float slopes fail, and why GCD-reduced tuples work

import math

# Float slope problem: 1/3 == 2/6 should be True, but floating point is lossy
print("Float slope demo:")
print(f"  1/3 == 2/6 as floats: {1/3 == 2/6}")            # True (luckily, but fragile)
print(f"  (1e15+1)/(3e15) vs 1/3: {(1e15+1)/(3e15) == 1/3}")  # False — precision loss

def slope_as_tuple(dy, dx):
    if dx == 0:
        return (1, 0)         # vertical — canonical representation
    if dy == 0:
        return (0, 1)         # horizontal — canonical representation
    g = math.gcd(abs(dy), abs(dx))
    dy, dx = dy // g, dx // g
    if dx < 0:                # normalize sign: dx always positive
        dy, dx = -dy, -dx
    return (dy, dx)

print("\nGCD-reduced slope tuples:")
print(f"  slope (2,4)   → {slope_as_tuple(2, 4)}")        # (1,2)
print(f"  slope (3,6)   → {slope_as_tuple(3, 6)}")        # (1,2) same slope!
print(f"  slope (-1,-2) → {slope_as_tuple(-1, -2)}")     # (1,2) same slope!
print(f"  slope (1,0)   → {slope_as_tuple(1, 0)}")        # (1,0) vertical

print("\nFast modular power:")
print(f"  2^1000 mod (10^9+7) = {pow(2, 1000, 10**9+7)}")
print(f"  Normal 2^1000 digit count = {len(str(2**1000))} digits (too big without mod)")

print("\nGCD Euclidean steps on (48, 18):")
a, b = 48, 18
while b:
    print(f"  gcd({a:3d}, {b:3d}) → gcd({b:3d}, {a%b:3d})")
    a, b = b, a % b
print(f"  result = {a}")

<a id='4'></a>
## 4. 🗂️ Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                         WHAT TO USE
──────────────────────────────────────────────────────────────────
"count primes up to n"                        Sieve of Eratosthenes O(n log log n)
"is n prime?"                                 Trial division O(sqrt(n))
"can we reach target with 2 container sizes"  Bezout (gcd divides target)
"points on same line" / slope grouping        GCD-reduced fraction as key
"nth column in spreadsheet" / modified base   (n-1) % base + offset trick
"a^b mod p" for large b                        pow(a, b, p) — built-in fast power
"combination nCr mod p"                       Lucas theorem / mod inverse
"repeated operation — find cycle"             Floyd's or math.gcd (Pisano period)
──────────────────────────────────────────────────────────────────

KEY DIVISIBILITY FACTS:
  gcd(a, 0) = a
  gcd(a, b) = gcd(b, a % b)        Euclidean
  lcm(a, b) = a * b // gcd(a, b)
  Bezout: ax + by = gcd(a, b)      always has integer solution
  Target reachable with jugs x,y:  target % gcd(x,y) == 0 AND target <= x+y

SLOPE CANONICAL FORM:
  dy, dx = y2-y1, x2-x1
  g = gcd(|dy|, |dx|)
  key = (dy//g, dx//g) with dx always positive (normalize sign)
  Special: dx==0 → key=(1,0) vertical; dy==0 → key=(0,1) horizontal
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Sieve of Eratosthenes — LC 204

---

```
PROBLEM:  Count the number of prime numbers less than n.
APPROACH: Boolean array is_prime[0..n-1], initialized True.
          For each p starting at 2: if still True, mark all multiples False.
          Start marking at p² (all smaller multiples were marked by smaller primes).
          Stop outer loop when p² >= n.

SLOW MOTION TRACE on n=20:

  init:  is_prime = [F,F,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T,T]
         (index)    [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]

  p=2:   mark [4,6,8,10,12,14,16,18] = False  (start at 4=2²)
  p=3:   mark [9,15] = False               (6 already False; start at 9=3²)
  p=4:   is_prime[4]=False → skip
  p=5:   5²=25>20 → STOP

  is_prime: [F,F,T,T,F,T,F,T,F,F,F,T,F,T,F,F,F,T,F,T]
  Primes:   [      2,3,  5,  7,        11,  13,        17,  19]
  Count = 8 ✓

KEY INSIGHT: Starting at p² is the crucial optimization. Every composite p*k
             where k < p has already been crossed by k's factor, which is smaller than p.
TIME:  O(n log log n) — sum of 1/p for all primes p (diverges slowly)
SPACE: O(n) — boolean array
```

In [ ]:
def count_primes(n):
    """
    LC 204 — Count Primes
    Approach: Sieve of Eratosthenes — boolean array, cross multiples starting at p².
    Args:
        n (int): count primes strictly less than n.
    Returns:
        int: number of primes in range [2, n-1].
    Time:  O(n log log n) — harmonic sum of prime reciprocals
    Space: O(n) — boolean sieve array
    """
    if n <= 2:
        return 0         # no primes exist below 2

    is_prime = [True] * n
    is_prime[0] = False   # 0 is not prime
    is_prime[1] = False   # 1 is not prime

    p = 2
    while p * p < n:       # stop when p² exceeds n — no new composites to mark
        if is_prime[p]:
            # start at p² — all p*k for k<p were already crossed by k's prime factor
            for j in range(p * p, n, p):
                is_prime[j] = False
        p += 1

    return sum(is_prime)   # count True entries = primes in [0..n-1]

# Slow motion on n=20:
# p=2: cross 4,6,8,10,12,14,16,18 (start at 4=2²)
# p=3: cross 9,15 (start at 9=3²; 6 already crossed)
# p=4: is_prime[4]=False → skip
# p=5: 5²=25≥20 → loop ends
# count True in is_prime[0..19] = 8 (primes: 2,3,5,7,11,13,17,19)

def test_harness(fn):
    tests = [
        (10, 4),    # primes < 10: 2,3,5,7
        (20, 8),    # primes < 20: 2,3,5,7,11,13,17,19
        (0, 0),     # no primes below 0
        (1, 0),     # no primes below 1
        (2, 0),     # no primes below 2
        (3, 1),     # primes < 3: just 2
        (100, 25),  # well-known: 25 primes below 100
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(count_primes)
print("count_primes defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Max Points on a Line — LC 149

---

```
PROBLEM:  Given n points in a 2D plane, find the maximum number of points
          that lie on the same straight line.
APPROACH: For each point as origin, compute the canonical slope to every other point.
          Slope = (dy, dx) reduced by gcd, with sign normalized (dx always ≥ 0).
          Count duplicates of each slope using a dict. Max count + 1 (for origin).
          Handle duplicate points (same coordinate as origin) separately.

SLOW MOTION TRACE on points = [[1,1],[2,2],[3,3],[3,1]]:

  Origin = [1,1]:
    to [2,2]: dy=1, dx=1, gcd=1 → key=(1,1)
    to [3,3]: dy=2, dx=2, gcd=2 → key=(1,1)  same slope!
    to [3,1]: dy=0, dx=2, gcd=2 → key=(0,1)  horizontal
    slope_count = {(1,1):2, (0,1):1}
    max slope = 2, max points through [1,1] = 2+1 = 3  (include origin)

  Origin = [2,2]:
    to [1,1]: dy=-1, dx=-1 → normalize dx>0: dy=1,dx=1 → key=(1,1)
    to [3,3]: dy=1,  dx=1  → key=(1,1)
    to [3,1]: dy=-1, dx=1  → key=(-1,1)
    max slope = 2 → max points = 3

  Overall max = 3 ✓

KEY INSIGHT: Slope as float fails due to precision. GCD-reduced fraction as tuple
             is an exact, hashable canonical form. Normalize sign so (1,2) and
             (-1,-2) map to the same key.
TIME:  O(n²) — for each of n origins, compute n slopes
SPACE: O(n) — slope dict per origin
```

In [ ]:
import math
from collections import defaultdict

def max_points(points):
    """
    LC 149 — Max Points on a Line
    Approach: for each origin, compute GCD-reduced slope to all others; count max slope group.
    Args:
        points (List[List[int]]): 2D coordinates.
    Returns:
        int: max number of collinear points.
    Time:  O(n²) — n origins × n slope computations
    Space: O(n)  — slope dict per origin (rebuilt each iteration)
    """
    n = len(points)
    if n <= 2:
        return n    # any 2 points define a line

    overall_max = 2

    for i in range(n):
        x1, y1 = points[i]
        slope_count = defaultdict(int)
        duplicates = 0   # same coordinate as origin — sits on EVERY line through origin

        for j in range(i + 1, n):
            x2, y2 = points[j]
            dx = x2 - x1
            dy = y2 - y1

            if dx == 0 and dy == 0:
                duplicates += 1    # same point — add to every line through origin
                continue

            g = math.gcd(abs(dy), abs(dx))
            dy, dx = dy // g, dx // g

            if dx < 0:             # normalize sign — dx always positive
                dy, dx = -dy, -dx
            elif dx == 0:
                dy = 1             # vertical: canonical (1, 0)

            slope_count[(dy, dx)] += 1

        # max in slope_count = most collinear points besides origin
        # +1 for origin itself, +duplicates (they sit on every line)
        if slope_count:
            local_max = max(slope_count.values()) + 1 + duplicates
        else:
            local_max = 1 + duplicates   # all points are same as origin

        overall_max = max(overall_max, local_max)

    return overall_max

# Slow motion on [[1,1],[2,2],[3,3],[3,1]]:
# origin (1,1): slopes {(1,1):2, (0,1):1} → max=2 → local_max=3
# origin (2,2): slopes {(1,1):2, (-1,1):1} → max=2 → local_max=3
# origin (3,3): slopes {(1,1):2, (0,-1):1} → max=2 → local_max=3
# origin (3,1): slopes {(1,1):1,(0,1):1,(-1,1):1} → max=1 → local_max=2
# overall_max = 3

def test_harness(fn):
    tests = [
        ([[1,1],[2,2],[3,3]], 3),
        ([[1,1],[3,2],[5,3],[4,1],[2,3],[1,4]], 4),
        ([[0,0]], 1),
        ([[0,0],[0,0]], 2),                    # duplicate points
        ([[1,1],[2,2],[3,3],[3,1]], 3),
        ([[0,0],[1,0],[2,0]], 3),              # horizontal line
        ([[0,0],[0,1],[0,2]], 3),              # vertical line
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(max_points)
print("max_points defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Water and Jug Problem — LC 365

---

```
PROBLEM:  Two jugs with capacity x and y liters. Can you measure exactly
          targetCapacity liters using only fills, emptying, and pouring between jugs?
APPROACH: Bezout's identity:
          ax + by = c has an integer solution ↔ gcd(x, y) divides c.
          Any volume reachable with the two jugs is a multiple of gcd(x, y).
          Also, target must be ≤ x + y (can't hold more than both jugs total).

SLOW MOTION TRACE:

  x=3, y=5, target=4:
    gcd(3, 5) = 1   →   1 divides every integer
    target=4 ≤ 3+5=8 ✓
    Answer: True

    Physical path:
    Fill 5L jug → pour into 3L jug → empty 3L → pour 2L remainder into 3L
    Fill 5L jug → pour into 3L jug (only 1L space) → 5L jug has 4L ✓

  x=2, y=6, target=5:
    gcd(2, 6) = 2   →   2 does NOT divide 5
    Answer: False  (can only reach even multiples: 0,2,4,6,8)

  x=1, y=2, target=3:
    gcd(1, 2) = 1   →   1 divides 3
    target=3 ≤ 1+2=3 ✓
    Answer: True  (fill both = 3L)

  x=2, y=4, target=2:
    gcd(2, 4) = 2   →   2 divides 2
    target=2 ≤ 2+4=6 ✓
    Answer: True  (just fill the 2L jug)

KEY INSIGHT: Bezout's theorem collapses a BFS graph problem into a single
             arithmetic check. No simulation needed.
TIME:  O(log(min(x,y))) — GCD computation only
SPACE: O(1)
```

In [ ]:
import math

def can_measure_water(x, y, target):
    """
    LC 365 — Water and Jug Problem
    Approach: Bezout's theorem — target reachable iff gcd(x,y) divides target AND target <= x+y.
    Args:
        x (int): capacity of jug 1.
        y (int): capacity of jug 2.
        target (int): desired measurement.
    Returns:
        bool: True if target is measurable.
    Time:  O(log min(x,y)) — Euclidean GCD
    Space: O(1)
    """
    if target > x + y:
        return False     # can't hold more water than both jugs combined

    if x == 0 or y == 0:
        return target == 0 or target == x + y  # only reachable: 0 or full

    # Bezout: target is reachable iff gcd(x, y) divides target
    return target % math.gcd(x, y) == 0

# Why this works — expanded explanation in comments:
# Every pour operation creates linear combination: a*x + b*y = current volume
# where a, b are integers (positive=fill, negative=empty that many times)
# Bezout's theorem: the set of all integers expressible as a*x+b*y
#   = {k * gcd(x,y) : k is any integer}
# So target is reachable iff gcd(x,y) | target
# The additional constraint: target must physically fit (target <= x+y)

def test_harness(fn):
    tests = [
        (3, 5, 4, True),     # gcd=1, 1|4, 4<=8 ✓
        (2, 6, 5, False),    # gcd=2, 2∤5
        (1, 2, 3, True),     # gcd=1, target=x+y=3 ✓
        (2, 4, 2, True),     # gcd=2, 2|2, 2<=6 ✓
        (3, 5, 0, True),     # target=0 always reachable (empty both)
        (3, 5, 9, False),    # target=9 > x+y=8
        (0, 0, 0, True),     # both empty, target 0
        (0, 0, 1, False),    # nothing to fill
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | x={inputs[0]},y={inputs[1]},target={inputs[2]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(can_measure_water)
print("can_measure_water defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Excel Sheet Column Title — LC 168

---

```
PROBLEM:  Given a column number (1-indexed), return its Excel column title.
          1→'A', 2→'B', ..., 26→'Z', 27→'AA', 28→'AB', ..., 702→'ZZ'
APPROACH: This is base-26 conversion, but there is NO zero digit.
          A=1, B=2, ..., Z=26 (not A=0 as you'd expect from base-26).
          Fix: subtract 1 before each modulo to shift range from [1..26] to [0..25].
          Then: digit = (n-1) % 26 → chr(ord('A') + digit)
                n = (n-1) // 26    → shift right by one digit
          Repeat until n == 0.

SLOW MOTION TRACE on n=701:

  Step 1: (701-1) % 26 = 700 % 26 = 24 → chr('A'+24) = 'Y'  n=(700)//26=26
  Step 2: (26-1)  % 26 = 25 % 26 = 25  → chr('A'+25) = 'Z'  n=(25)//26=0
  n==0 → stop, reverse digits: 'ZY' → "ZY"  ✓

  Verify: Z=26, ZY = 26*26 + 25 = 676+25 = 701 ✓

SLOW MOTION TRACE on n=28:

  Step 1: (28-1) % 26 = 27 % 26 = 1 → chr('A'+1) = 'B'  n=(27)//26=1
  Step 2: (1-1)  % 26 = 0 % 26 = 0  → chr('A'+0) = 'A'  n=(0)//26=0
  n==0 → stop, reverse: 'BA' → "AB" ✓

KEY INSIGHT: The (n-1) shift is the entire trick. Standard base-26 fails
             because there is no 'zero' digit — Z means 26, not 0.
             After shifting, the math is identical to base conversion.
TIME:  O(log_26(n)) — number of digits in base-26 representation
SPACE: O(log_26(n)) — result string length
```

In [ ]:
def convert_to_title(columnNumber):
    """
    LC 168 — Excel Sheet Column Title
    Approach: modified base-26 with (n-1) shift to handle no-zero-digit alphabet.
    Args:
        columnNumber (int): 1-indexed column number.
    Returns:
        str: Excel column title (e.g., 1→'A', 27→'AA').
    Time:  O(log_26(n)) — digits in base-26 representation
    Space: O(log_26(n)) — result string
    """
    result = []
    n = columnNumber

    while n > 0:
        n -= 1                              # shift: move [1..26] to [0..25]
        digit = n % 26                      # extract least significant base-26 digit
        result.append(chr(ord('A') + digit))  # map 0→'A', 1→'B', ..., 25→'Z'
        n //= 26                            # shift right: discard processed digit

    return ''.join(reversed(result))        # digits built LSB-first, reverse to get MSB-first

# Slow motion on n=701:
# iter 1: n=700, digit=700%26=24 → 'Y', n=700//26=26
# iter 2: n=25,  digit=25%26=25  → 'Z', n=25//26=0
# result = reverse(['Y','Z']) = 'ZY' ✓

# Slow motion on n=52:
# iter 1: n=51, digit=51%26=25 → 'Z', n=51//26=1
# iter 2: n=0,  digit=0%26=0   → 'A', n=0//26=0
# result = reverse(['Z','A']) = 'AZ' ✓ (AZ = 1*26 + 26 = 52)

def test_harness(fn):
    tests = [
        (1, 'A'),
        (26, 'Z'),
        (27, 'AA'),
        (28, 'AB'),
        (52, 'AZ'),
        (701, 'ZY'),
        (702, 'ZZ'),
        (703, 'AAA'),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(convert_to_title)
print("convert_to_title defined.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Fast Power & Modular Arithmetic

---

```
PROBLEM:  Compute base^exp (mod m) efficiently. Also: implement pow(x, n) for
          LC 50 which allows negative exponents and must handle n as 64-bit int.
APPROACH: Binary exponentiation (fast power):
          If exp is even: base^exp = (base²)^(exp//2)   → halve the exponent
          If exp is odd:  base^exp = base * base^(exp-1) → peel one factor
          Repeat until exp == 0. This takes only O(log exp) multiplications.

SLOW MOTION TRACE on 2^10:

  fast_pow(2, 10):
    exp=10 even: (2²)^5   = fast_pow(4, 5)
    exp=5  odd:  4 * 4^4  = 4 * fast_pow(16, 4)
    exp=4  even: (16²)^2  = fast_pow(256, 2)
    exp=2  even: (256²)^1 = fast_pow(65536, 1)
    exp=1  odd:  65536 * 65536^0 = fast_pow(65536^2, 0)
    exp=0: return 1
    unwind: 1 * 65536 = 65536 → but wait, we need to unwind carefully

  Iterative approach (cleaner for interviews):
    result=1, base=2, exp=10
    exp=10(even): base=2²=4,   exp=5
    exp=5 (odd):  result*=4=4, base=4²=16, exp=2
    exp=2 (even): base=16²=256,exp=1
    exp=1 (odd):  result*=256=1024, base=256²,exp=0
    exp=0: return 1024  ✓ (2^10=1024)

KEY INSIGHT: Each iteration halves the exponent. After log₂(exp) iterations
             we reach exp=0. Same principle as binary search.
             For mod: take mod at each multiplication to prevent overflow.
TIME:  O(log exp) — halve exponent each step
SPACE: O(1) iterative, O(log exp) recursive stack
```

In [ ]:
def my_pow(x, n):
    """
    LC 50 — Pow(x, n)
    Approach: iterative binary exponentiation; handle negative n by inverting x.
    Args:
        x (float): base.
        n (int): exponent (can be negative, 64-bit range).
    Returns:
        float: x^n.
    Time:  O(log |n|) — halve exponent each iteration
    Space: O(1)       — iterative, no recursion stack
    """
    if n < 0:
        x = 1 / x    # negative exponent → invert base, work with positive exponent
        n = -n

    result = 1.0
    base = x

    while n > 0:
        if n % 2 == 1:        # odd exponent: peel off one factor
            result *= base
        base *= base          # square the base — absorb one power-of-2 worth of exponent
        n //= 2               # halve the exponent

    return result


def mod_pow(base, exp, mod):
    """
    Modular fast power: base^exp mod m.
    Approach: binary exponentiation with mod at each step to prevent integer overflow.
    Args:
        base (int): base value.
        exp (int): non-negative exponent.
        mod (int): modulus.
    Returns:
        int: base^exp % mod.
    Time:  O(log exp)
    Space: O(1)
    """
    result = 1
    base %= mod              # ensure base is within mod range from the start
    while exp > 0:
        if exp % 2 == 1:
            result = result * base % mod   # take mod after each multiply
        base = base * base % mod           # square base, keep in mod range
        exp //= 2
    return result

# Slow motion on my_pow(2, 10):
# n=10(even): result=1,  base=4,    n=5
# n=5 (odd):  result=4,  base=16,   n=2
# n=2 (even): result=4,  base=256,  n=1
# n=1 (odd):  result=1024,base=..., n=0
# return 1024.0 ✓

def test_harness(fn):
    tests = [
        (2.0, 10, 1024.0),
        (2.1, 3, round(2.1**3, 5)),
        (2.0, -2, 0.25),
        (2.0, 0, 1.0),
        (1.0, 1000000, 1.0),
        (-2.0, 3, -8.0),
        (-2.0, 2, 4.0),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = round(fn(*inputs), 5)
        exp_r = round(expected, 5)
        status = "PASSED" if abs(got - exp_r) < 1e-4 else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={exp_r} | got={got}")
        passed += (status == "PASSED")
    print(f"{passed}/{len(tests)} tests passed")

    print()
    print("mod_pow demos:")
    print(f"  2^100 mod (10^9+7)  = {mod_pow(2, 100, 10**9+7)}")
    print(f"  Python pow(2,100,m) = {pow(2, 100, 10**9+7)}  (should match)")
    print(f"  3^1000 mod 13       = {mod_pow(3, 1000, 13)}")

test_harness(my_pow)
print("my_pow and mod_pow defined.")

<a id='10'></a>
## 10. 🗺️ The Math & Number Theory Decision Map

```
QUESTION TYPE                    KEY TECHNIQUE              LC PROBLEMS
────────────────────────────────────────────────────────────────────────────
Count primes < n                 Sieve of Eratosthenes      204
Max collinear points             GCD-reduced slope fraction  149
Jug / container reachability     Bezout: target%gcd(x,y)==0 365
Base conversion (no zero digit)  (n-1) % base shift trick    168
Large exponent computation       Binary fast power           50
Large exponent mod m             pow(base, exp, mod)         (built-in)
────────────────────────────────────────────────────────────────────────────

GCD CHEAT SHEET:
  gcd(a, 0) = a
  gcd(0, b) = b
  gcd(a, b) = gcd(b, a%b)           Euclidean algorithm O(log min)
  lcm(a, b) = a * b // gcd(a, b)   watch for overflow: use a//gcd * b

BEZOUT CONDITIONS:
  ax + by = target solvable ↔ gcd(a,b) divides target
  Jug problem: also check target <= x+y

SIEVE OPTIMIZATION:
  Only cross multiples of PRIMES (skip composite p)
  Start at p² (all p*k for k<p already crossed)
  Stop outer loop at sqrt(n)

SLOPE CANONICAL FORM:
  g = gcd(|dy|, |dx|)
  key = (dy//g, dx//g) with dx always positive
  Special: dx=0 → key=(1,0); dy=0 → key=(0,1)
  NEVER use float slopes — use exact fraction tuple
```

<a id='11'></a>
## 11. 📋 Interview Cheat Sheet

### When to reach for Math & Number Theory:

| Signal | What to Do |
|--------|------------|
| "count primes" | Sieve: cross multiples starting at p² |
| "points on a line" | GCD slope fraction as dict key |
| "can we reach target with 2 sizes" | Bezout: target % gcd == 0 |
| "Excel column" / modified base | (n-1) % 26 shift trick |
| "a^b mod p" for huge b | pow(a, b, p) — Python built-in |
| "is n prime" | trial division up to sqrt(n) |

### The O(log n) operations — memorize these:

```python
import math
math.gcd(a, b)          # O(log min) Euclidean
a * b // math.gcd(a,b)  # LCM
pow(base, exp, mod)     # O(log exp) modular fast power — built-in
target % math.gcd(x,y) == 0  # Bezout reachability check
```

### Common templates:

```python
# TEMPLATE 1: SIEVE OF ERATOSTHENES
def sieve(n):
    is_p = [True] * n
    is_p[0] = is_p[1] = False
    p = 2
    while p * p < n:
        if is_p[p]:
            for j in range(p*p, n, p): is_p[j] = False
        p += 1
    return [i for i in range(n) if is_p[i]]

# TEMPLATE 2: GCD-REDUCED SLOPE TUPLE
def slope_key(dy, dx):
    if dx == 0: return (1, 0)      # vertical
    if dy == 0: return (0, 1)      # horizontal
    g = math.gcd(abs(dy), abs(dx))
    dy, dx = dy//g, dx//g
    if dx < 0: dy, dx = -dy, -dx  # normalize sign
    return (dy, dx)

# TEMPLATE 3: MODIFIED BASE CONVERSION (no-zero-digit)
def convert(n, base=26, offset=ord('A')):
    result = []
    while n > 0:
        n -= 1
        result.append(chr(offset + n % base))
        n //= base
    return ''.join(reversed(result))

# TEMPLATE 4: FAST POWER (iterative)
def fast_pow(base, exp, mod=None):
    result = 1
    while exp > 0:
        if exp & 1: result = result * base if mod is None else result * base % mod
        base = base * base if mod is None else base * base % mod
        exp >>= 1
    return result
```

### Gotchas to not forget:

```
❌  float slope = (y2-y1)/(x2-x1) — precision loss gives wrong groupings
✅  slope_key = (dy//gcd, dx//gcd) tuple with normalized sign
❌  sieve: start crossing at 2*p instead of p² — O(n log n) instead of O(n log log n)
✅  sieve: for j in range(p*p, n, p) — start at p²
❌  Bezout: only check target%gcd==0 without target<=x+y — can return True for target=100 with x=1,y=2
✅  both: target%gcd==0 AND target<=x+y
❌  Excel base-26: n%26 for digits — fails because there is no zero digit
✅  Excel: n-=1 FIRST, then n%26, then n//=26
❌  gcd(0, 0) → returns 0, which causes division by zero later — guard if needed
✅  check for zero inputs before using gcd in divisions
```

<a id='12'></a>
## 12. 🗺️ Summary Map

```
              MATH & NUMBER THEORY
                        │
         ┌──────────────┼──────────────┐
         │              │              │
      PRIMES           GCD           BASE
         │              │           CONVERT
      Sieve of       Euclidean         │
      Eratosthenes   Algorithm      (n-1)%26
      O(n log log n) O(log min)     shift trick
      LC 204                        LC 168
                     │
            ┌────────┴────────┐
          SLOPE             BEZOUT
          FRACTION          THEOREM
             │                 │
         (dy//g,            target %
          dx//g)            gcd == 0
         LC 149              LC 365

      FAST POWER
      Binary exponentiation
      O(log exp)
      pow(base, exp, mod)  ← Python built-in
      LC 50

THE UNIVERSAL RULE:
  Number theory problems hide in plain sight.
  When you see "can we reach X" or "are these the same" or "count structure" —
  ask: is there a GCD relationship? a mod pattern? a base-conversion structure?
  The math answer is always O(log n). The BFS simulation is always O(n²).
```

---
*End of Math & Number Theory Master Guide — Sean Edition*